### Hypothesis Testing 
In this notebook we will utilize statistical learning methods to determine whether or not the input variables (X) are effective in explaining the predictor (Y). To do this, we must compute the standard errors of the coeficent for each input and T-test it to determine if the coeficent is statisticaly significant. Our chosen model for this phase will be K-Nearest Neighbors and Logistic Regression. These simple models outputted the best evalution results out of all the different algorithms that we're fitted and tested. 

In [13]:
# new dataset with updated features 
from pathlib import Path  
import pandas as pd 

# instantiation process 
dir = Path.cwd()

# find the data file 
data_path = dir.parent / 'Data' / 'significant_feat.csv'

# load data in object 
df = pd.read_csv(data_path)

# view result 
df.head()


,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked_644,Embarked_C,Embarked_Q,Sex_female,Cabin_encoded,interaction4,interaction5,interaction6,interaction7,Ratio1,Ratio4,Ratio5
0,0,0.827377,-0.581659,0.432793,0,-0.879741,False,False,False,False,0.303217,0.511709,-0.727877,-0.000000,-0.879741,0.661171,-0.000000,-1.136699
1,1,-1.566107,0.649327,0.432793,0,1.361220,False,True,False,True,0.382889,0.883877,-2.131816,1.361220,0.000000,0.477018,0.734635,0.000000
2,1,0.827377,-0.273913,-0.474545,0,-0.798540,False,False,False,True,0.298863,0.218730,-0.660694,-0.798540,-0.000000,0.343017,-1.252285,-0.000000
3,1,-1.566107,0.418517,0.432793,0,1.062038,False,False,False,True,0.000000,0.444481,-1.663265,1.062038,0.000000,0.394070,0.941586,0.000000
4,0,0.827377,0.418517,-0.474545,0,-0.784179,False,False,False,False,0.303215,-0.328192,-0.648812,-0.000000,-0.784179,-0.533701,-0.000000,-1.275219


In [14]:
# training phase 
# Peforming hyptothesis testing on a training set is key in identify which features are significant 
from sklearn.model_selection import train_test_split

# inputs/features (drop target feature to avoid data leakage)
X = df.drop(columns=['Survived'])

# Target variable (Save target variable into data as an csv for future downstream work)
Y = df['Survived']

# define where you want it to go and save it 
save_path = Path.cwd().parent / 'Data' / 'target.csv'
Y.to_csv(save_path, index = False)

# 70/30 split 
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size= 0.3, shuffle= True, random_state= 42)

#Reset indexs 
X_train = X_train.reset_index(drop = True)
Y_train = Y_train.reset_index(drop = True)

# make data 1D (data will load with index)
Y_train = Y_train.values.ravel()


# K-Nearest Neighbors
Important Insight: Since KNN does not have p-values because it is a non-parametric model, we will use sequential feature selection which automatically handles the backward elimintation and determines the best input features by checking accuaracy/precision at each step. 


In [15]:
from sklearn.feature_selection import SequentialFeatureSelector # elimination process
from sklearn.neighbors import KNeighborsClassifier # model 

# initialze model with the gridsearch adjusted parameters 
knn = KNeighborsClassifier(leaf_size=10, weights='distance')

# backward elimination for KKN 
sfs = SequentialFeatureSelector(knn, 
                                n_features_to_select = 'auto', 
                                direction = 'backward', 
                                scoring = 'precision')
# fit data onto model 
sfs.fit(X_train, Y_train)

# best features
selected_features = X_train.columns[sfs.get_support()]

# results
selected_features

Index(['Pclass', 'Parch', 'Fare', 'Embarked_C', 'Embarked_Q', 'interaction6',
       'interaction7', 'Ratio4', 'Ratio5'],
      dtype='object')

# Logit testing 
Important Insight: Redundancy is prevelant between the interaction variables, causing the model to suffer when predicting the raw column fare which is made up of the engieneered features. Also, removing the redundant categories like male and embarked to allow the model to use one category as the baseline. 


In [16]:
import statsmodels.api as sm 

#convert bool objects to floats (statsmodel has a hard time seeing them as bool objects)
X_train_numeric = X_train.astype(float)
                    
# manually add a constant intercept to the data 
X_train_const = sm.add_constant(X_train_numeric)

# fit the model
model = sm.Logit(Y_train, X_train_const)
result = model.fit(maxiter = 1000)

# view results 
print(f'Summary {result.summary()}')

         Current function value: 0.434344
         Iterations: 1000
Summary                            Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  623
Model:                          Logit   Df Residuals:                      606
Method:                           MLE   Df Model:                           16
Date:                Thu, 23 Apr 2026   Pseudo R-squ.:                  0.3413
Time:                        14:25:41   Log-Likelihood:                -270.60
converged:                      False   LL-Null:                       -410.79
Covariance Type:            nonrobust   LLR p-value:                 2.896e-50
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -2.2660      0.305     -7.426      0.000      -2.864      -1.668
Pclass           -0.4719      0.178     -2.648

C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


# Variance Inflation Factor
Important Insight: 3 of the input variables posted infinite outputs, flagging mathematical copies of each other. 1 input variable is right on the edge of being to high, meaning it's mostly redundant. These inputs will be futher removed from the dataset and hypthesis training will be conducted again to view any updated changes to the models performance. 

In [17]:
# variance inflation error
from statsmodels.stats.outliers_influence import variance_inflation_factor

# ensure X has a constant intercept 
X = X_train_const.copy()

# calculate VIFs
vifs = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# create table 
pd.DataFrame({'feature' : X.columns, 'VIF': vifs})


C:\Users\dmiracju\AppData\Roaming\Python\Python313\site-packages\statsmodels\stats\outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


,feature,VIF
0,const,7.045269
1,Pclass,2.711776
2,Age,1.311714
3,SibSp,1.837516
4,Parch,1.614003
5,Fare,inf
6,Embarked_644,1.032126
7,Embarked_C,1.139850
8,Embarked_Q,1.135024
9,Sex_female,1.194654


# Logit Final Testing 
Important Insight: The Model has converged and after the removal of all non-statisically significant features, the results are great!  



In [18]:
# remove redundant columns 
X_train_const = X_train_const.drop(columns = ['Fare', 'interaction4', 'interaction5', 'interaction6', 'interaction7', 
                                              'Embarked_644', 'Embarked_C', 'Ratio4', 'Ratio5', 'Ratio1', 'Parch', 'Embarked_Q'])

# begin model fitting 
model = sm.Logit(Y_train, X_train_const)
results = model.fit(maxiter = 2000)

# results
print(f'Summary {results.summary()}')

Optimization terminated successfully.
         Current function value: 0.450907
         Iterations 6
Summary                            Logit Regression Results                           
Dep. Variable:                      y   No. Observations:                  623
Model:                          Logit   Df Residuals:                      617
Method:                           MLE   Df Model:                            5
Date:                Thu, 23 Apr 2026   Pseudo R-squ.:                  0.3162
Time:                        14:25:41   Log-Likelihood:                -280.92
converged:                       True   LL-Null:                       -410.79
Covariance Type:            nonrobust   LLR p-value:                 4.446e-54
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -2.1794      0.271     -8.052      0.000      -2.710      -1.649
Pclass      

# Dataset Creation 
- Important Insight: Below are the curated dataset preferences designed for simple and complex models. Each dataset is specific to a model base on the performance or the necessary data structure needed to bring out the model best predicitions without the increase of noise. 


In [20]:
# Drop the uncessary features needed to improve the KNN Model 
# This will also be considered the non-linear dataset
non_linear_df = df[['Pclass', 'Age', 'Parch', 'Fare', 'Embarked_C', 'interaction4',
       'interaction5', 'interaction7', 'Ratio4']]

# drop the uncessary features needed to improve the logistic Regression Model
# This will also be considered the linear dataset 
linear_df = df[['Pclass', 'Age', 'SibSp', 'Sex_female', 'Cabin_encoded']]

# save datasets to data directory
save_path = dir.parent / 'Data' / 'linear.csv'
save_path1 = dir.parent / 'Data' / 'non_linear.csv'

# linear csv
linear_df.to_csv(save_path, index = False)
# non_linear.csv 
non_linear_df.to_csv(save_path1, index = False)

